# ==============================================================================
# DRONE SWARM THEATER THREAT SIMULATION & TRAJECTORY PLANNING
# ==============================================================================
# This notebook models staggered, low-altitude, waypoint-splined drone swarms
# (SEAD Radar-Hunters, Saturation Decoys, and Loitering Strike Munitions) operating
# against an integrated theater defense network (SAM + SHORAD/CIWS).
#
# Trajectory kinematics are adapted from continuous 3D polynomial/spline trajectory
# generation principles (inspired by continuous C2 trajectory state formulations)
# and include an interactive Play/Pause 3D visualizer with staggered attack waves.


In [1]:
# ==============================================================================
# 1. CORE IMPORTS & CONFIGURATION
# ==============================================================================
import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline
import plotly.graph_objects as go
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

np.random.seed(42)
print("Core simulation, spline trajectory, and interactive visualizer modules loaded.")


Core simulation, spline trajectory, and interactive visualizer modules loaded.


In [2]:
# ==============================================================================
# 2. CONTINUOUS 3D TRAJECTORY DYNAMICS ENGINE (ADAPTED FOR THEATER UAVs)
# ==============================================================================

@dataclass
class TrajectoryState:
    """Kinematic state representation at time t."""
    t: float
    position: np.ndarray      # [x, y, z] in meters
    velocity: np.ndarray      # [vx, vy, vz] in m/s
    acceleration: np.ndarray  # [ax, ay, az] in m/s^2
    speed: float              # scalar speed in m/s
    heading: float            # yaw heading angle in radians
    pitch: float              # climb/dive pitch angle in radians

class DroneSplineTrajectory:
    """
    Generates C2 continuous, smooth 3D trajectories across strategic waypoints
    using cubic spline interpolation, speed constraints, and altitude profiles.
    """
    def __init__(self, waypoints: np.ndarray, flight_duration: float, launch_time: float = 0.0):
        self.waypoints = np.array(waypoints, dtype=np.float64)  # shape (N, 3)
        self.launch_time = float(launch_time)
        self.flight_duration = float(flight_duration)
        self.impact_time = self.launch_time + self.flight_duration
        
        # Calculate cumulative segment distances to distribute waypoint time allocations
        diffs = np.diff(self.waypoints, axis=0)
        seg_lengths = np.linalg.norm(diffs, axis=1)
        total_dist = np.sum(seg_lengths)
        if total_dist < 1e-3:
            total_dist = 1.0
        
        cum_dist = np.insert(np.cumsum(seg_lengths), 0, 0.0)
        norm_dist = cum_dist / total_dist
        
        # Time distribution across waypoints
        self.waypoint_times = self.launch_time + norm_dist * self.flight_duration
        
        # Build cubic splines for smooth C2 trajectories in X, Y, Z
        self.cs_x = CubicSpline(self.waypoint_times, self.waypoints[:, 0], bc_type='natural')
        self.cs_y = CubicSpline(self.waypoint_times, self.waypoints[:, 1], bc_type='natural')
        self.cs_z = CubicSpline(self.waypoint_times, self.waypoints[:, 2], bc_type='natural')
        
    def evaluate(self, t: float) -> TrajectoryState:
        t_clamped = np.clip(t, self.launch_time, self.impact_time)
        
        pos = np.array([self.cs_x(t_clamped), self.cs_y(t_clamped), max(20.0, float(self.cs_z(t_clamped)))])
        vel = np.array([self.cs_x(t_clamped, 1), self.cs_y(t_clamped, 1), self.cs_z(t_clamped, 1)])
        acc = np.array([self.cs_x(t_clamped, 2), self.cs_y(t_clamped, 2), self.cs_z(t_clamped, 2)])
        
        speed = float(np.linalg.norm(vel))
        heading = float(np.arctan2(vel[1], vel[0])) if speed > 1e-3 else 0.0
        pitch = float(np.arcsin(np.clip(vel[2] / speed, -1.0, 1.0))) if speed > 1e-3 else 0.0
        
        return TrajectoryState(
            t=float(t),
            position=pos,
            velocity=vel,
            acceleration=acc,
            speed=speed,
            heading=heading,
            pitch=pitch
        )

print("Continuous 3D Spline Trajectory physics engine initialized.")


Continuous 3D Spline Trajectory physics engine initialized.


In [3]:
# ==============================================================================
# 3. THEATER GEOGRAPHY, SITES & DEFENDER BATTERY NETWORK
# ==============================================================================
THEATER_X_MAX = 12000e3  # 12,000 km downrange
THEATER_Y_MAX = 10000e3  # 10,000 km crossrange
THEATER_Z_MAX = 50e3     # 50 km local ceiling for low-altitude drones

# 5 Aggressor Launch Complexes (Western Sector: 0 - 500 km)
AGGRESSOR_LAUNCH_BASES = [
    {"id": "Base A1", "pos": np.array([100e3, 1500e3, 50.0])},
    {"id": "Base A2", "pos": np.array([250e3, 3500e3, 50.0])},
    {"id": "Base A3", "pos": np.array([150e3, 5000e3, 50.0])},
    {"id": "Base A4", "pos": np.array([300e3, 6500e3, 50.0])},
    {"id": "Base A5", "pos": np.array([200e3, 8500e3, 50.0])},
]

# 10 Vital Defending Targets in the Eastern Sector
DEFENDING_TARGETS = [
    {"id": 1, "name": "Early Warning Radar Array Alpha", "type": "Radar Grid", "pos": np.array([10200e3, 2200e3, 100.0]), "value": 150.0},
    {"id": 2, "name": "Strategic Command HQ Alpha", "type": "Strategic HQ", "pos": np.array([10500e3, 4800e3, 50.0]), "value": 180.0},
    {"id": 3, "name": "Theater Defense Battery HQ", "type": "SAM Defense", "pos": np.array([10800e3, 7600e3, 50.0]), "value": 130.0},
    {"id": 4, "name": "Central Radar Node Bravo", "type": "Radar Grid", "pos": np.array([11100e3, 3200e3, 100.0]), "value": 140.0},
    {"id": 5, "name": "Primary Airbase Victor", "type": "Military Runway", "pos": np.array([11300e3, 6000e3, 30.0]), "value": 110.0},
    {"id": 6, "name": "Forward Radar Picket Charlie", "type": "Radar Grid", "pos": np.array([9800e3, 5000e3, 80.0]), "value": 120.0},
    {"id": 7, "name": "Hardened Missile Silo Delta", "type": "Missile Silo", "pos": np.array([10700e3, 2000e3, 0.0]), "value": 140.0},
    {"id": 8, "name": "Tactical Fighter Runway Echo", "type": "Military Runway", "pos": np.array([11500e3, 4500e3, 30.0]), "value": 100.0},
    {"id": 9, "name": "Satellite Uplink Station", "type": "Radar Grid", "pos": np.array([11200e3, 8000e3, 120.0]), "value": 125.0},
    {"id": 10, "name": "Coastal Defense Runway Foxtrot", "type": "Military Runway", "pos": np.array([10100e3, 8200e3, 20.0]), "value": 105.0}
]

# 6 Defending SAM & Point Defense Batteries (Theater-calibrated ranges and interceptor kinetics)
DEFENDER_BATTERIES = [
    {"id": 1, "name": "Battery D1 (Long-Range SAM North)", "pos": np.array([10100e3, 2000e3, 50.0]), "type": "SAM", "magazine": 14, "radar_range": 1800e3, "interceptor_speed": 120000.0, "pk": 0.88, "min_rcs": 0.02},
    {"id": 2, "name": "Battery D2 (SHORAD/CIWS Central)", "pos": np.array([10400e3, 4800e3, 50.0]), "type": "SHORAD", "magazine": 30, "radar_range": 700e3, "interceptor_speed": 60000.0, "pk": 0.94, "min_rcs": 0.01},
    {"id": 3, "name": "Battery D3 (Long-Range SAM South)", "pos": np.array([10700e3, 7600e3, 50.0]), "type": "SAM", "magazine": 14, "radar_range": 1800e3, "interceptor_speed": 120000.0, "pk": 0.88, "min_rcs": 0.02},
    {"id": 4, "name": "Battery D4 (SHORAD North Radar Guard)", "pos": np.array([10200e3, 2300e3, 50.0]), "type": "SHORAD", "magazine": 24, "radar_range": 700e3, "interceptor_speed": 60000.0, "pk": 0.92, "min_rcs": 0.01},
    {"id": 5, "name": "Battery D5 (SAM Mid-Sector)", "pos": np.array([11000e3, 5500e3, 50.0]), "type": "SAM", "magazine": 16, "radar_range": 1600e3, "interceptor_speed": 115000.0, "pk": 0.85, "min_rcs": 0.02},
    {"id": 6, "name": "Battery D6 (SHORAD South Airbase Guard)", "pos": np.array([10150e3, 8100e3, 50.0]), "type": "SHORAD", "magazine": 24, "radar_range": 700e3, "interceptor_speed": 60000.0, "pk": 0.93, "min_rcs": 0.01}
]

print(f"Theater configured: {len(AGGRESSOR_LAUNCH_BASES)} launch bases, {len(DEFENDING_TARGETS)} vital targets, {len(DEFENDER_BATTERIES)} defense batteries.")


Theater configured: 5 launch bases, 10 vital targets, 6 defense batteries.


In [4]:
# ==============================================================================
# 4. STAGGERED DRONE SWARM ATTACK WAVES & WAYPOINT GENERATOR
# ==============================================================================

def generate_staggered_drone_swarm(n_drones=30, seed=101):
    """
    Generates a 3-wave staggered drone swarm:
    - Wave 1 (t_launch = 0 - 20s): SEAD Radar-Hunters (Flanking routes, stealth low-altitude)
    - Wave 2 (t_launch = 30 - 55s): Saturation Decoy Swarms (High volume center bait)
    - Wave 3 (t_launch = 65 - 90s): Loitering Strike Drones (Direct precision strike)
    """
    rng = np.random.RandomState(seed)
    drone_threats = []
    
    # 3 Staggered attack waves
    wave_configs = [
        {"wave": 1, "role": "SEAD Radar-Hunter", "color": "#ff007f", "rcs": 0.02, "alt": 120.0, "t_range": (0.0, 20.0), "flight_time": 140.0},
        {"wave": 2, "role": "Saturation Decoy Swarm", "color": "#ffa500", "rcs": 0.08, "alt": 280.0, "t_range": (30.0, 55.0), "flight_time": 130.0},
        {"wave": 3, "role": "Loitering Strike Drone", "color": "#00ffcc", "rcs": 0.03, "alt": 80.0, "t_range": (65.0, 90.0), "flight_time": 120.0},
    ]
    
    drones_per_wave = n_drones // 3
    
    threat_idx = 1
    for w_cfg in wave_configs:
        for _ in range(drones_per_wave):
            base = AGGRESSOR_LAUNCH_BASES[rng.choice(len(AGGRESSOR_LAUNCH_BASES))]
            
            # Targeting logic
            if w_cfg["role"] == "SEAD Radar-Hunter":
                radar_targets = [t for t in DEFENDING_TARGETS if "Radar" in t["type"]]
                target = radar_targets[rng.choice(len(radar_targets))]
                flank_choice = rng.choice(["north_flank", "south_flank"])
            elif w_cfg["role"] == "Saturation Decoy Swarm":
                target = DEFENDING_TARGETS[rng.choice(len(DEFENDING_TARGETS))]
                flank_choice = rng.choice(["center_corridor", "north_flank", "south_flank"])
            else:
                target = DEFENDING_TARGETS[rng.choice(len(DEFENDING_TARGETS))]
                flank_choice = rng.choice(["center_corridor", "south_flank", "north_flank"])
                
            p_start = base["pos"]
            p_end = target["pos"]
            
            # Generate waypoints based on corridor selection
            if flank_choice == "north_flank":
                wp1 = np.array([p_start[0] + 3000e3, min(THEATER_Y_MAX - 400e3, p_start[1] + 2500e3), w_cfg["alt"] + rng.uniform(-15, 15)])
                wp2 = np.array([8000e3, THEATER_Y_MAX - 500e3, w_cfg["alt"] + rng.uniform(-10, 10)])
                wp3 = np.array([p_end[0] - 300e3, p_end[1] + 150e3, w_cfg["alt"] + 40])
            elif flank_choice == "south_flank":
                wp1 = np.array([p_start[0] + 3000e3, max(400e3, p_start[1] - 2500e3), w_cfg["alt"] + rng.uniform(-15, 15)])
                wp2 = np.array([8000e3, 500e3, w_cfg["alt"] + rng.uniform(-10, 10)])
                wp3 = np.array([p_end[0] - 300e3, p_end[1] - 150e3, w_cfg["alt"] + 40])
            else: # Center corridor
                wp1 = np.array([p_start[0] + 3500e3, p_start[1] + rng.uniform(-400e3, 400e3), w_cfg["alt"]])
                wp2 = np.array([7500e3, (p_start[1] + p_end[1])/2 + rng.uniform(-300e3, 300e3), w_cfg["alt"]])
                wp3 = np.array([p_end[0] - 350e3, p_end[1] + rng.uniform(-60e3, 60e3), w_cfg["alt"] + 30])
                
            waypoints = np.array([p_start, wp1, wp2, wp3, p_end])
            
            t_launch = float(rng.uniform(w_cfg["t_range"][0], w_cfg["t_range"][1]))
            flight_time = float(w_cfg["flight_time"] + rng.uniform(-10.0, 10.0))
            
            trajectory = DroneSplineTrajectory(waypoints, flight_duration=flight_time, launch_time=t_launch)
            
            drone_threats.append({
                "id": threat_idx,
                "wave": w_cfg["wave"],
                "role": w_cfg["role"],
                "color": w_cfg["color"],
                "base": base["id"],
                "base_pos": p_start,
                "target": target["name"],
                "target_pos": p_end,
                "flank_route": flank_choice,
                "rcs": w_cfg["rcs"],
                "t_launch": t_launch,
                "flight_time": flight_time,
                "t_impact": t_launch + flight_time,
                "trajectory": trajectory
            })
            threat_idx += 1
            
    return drone_threats

drone_threats = generate_staggered_drone_swarm(n_drones=30)
print(f"Generated {len(drone_threats)} staggered drone threats across 3 tactical attack waves.")


Generated 30 staggered drone threats across 3 tactical attack waves.


In [5]:
# ==============================================================================
# 5. CLOSED-LOOP DRONE INTERCEPTION & DEFENSE ENGAGEMENT SIMULATOR
# ==============================================================================

def simulate_drone_swarm_engagement(threats, batteries, targets):
    duel_results = []
    
    battery_states = [dict(b) for b in batteries]
    for b in battery_states:
        b["shots_fired"] = 0
        b["intercepts_confirmed"] = 0
        
    target_status = {t["name"]: {"destroyed": False, "hits": 0, "value": t["value"]} for t in targets}
    
    # Process threats in chronological order of impact
    for threat in sorted(threats, key=lambda x: x["t_impact"]):
        traj = threat["trajectory"]
        t_launch = threat["t_launch"]
        t_impact = threat["t_impact"]
        
        sample_times = np.linspace(t_launch, t_impact, 80)
        
        intercepted = False
        engaging_battery = None
        engaging_battery_pos = None
        t_intercept = None
        t_fire = None
        intercept_pos = None
        interception_method = None
        
        for t_curr in sample_times:
            state = traj.evaluate(t_curr)
            pos = state.position
            
            candidates = []
            for b in battery_states:
                if b["magazine"] <= 0:
                    continue
                # Radar detection threshold calculation
                if threat["rcs"] < b["min_rcs"]:
                    det_range = b["radar_range"] * (threat["rcs"] / 0.1)**0.25
                else:
                    det_range = b["radar_range"]
                    
                dist = np.linalg.norm(pos - b["pos"])
                if dist <= det_range:
                    candidates.append((b, dist))
                    
            if not candidates:
                continue
                
            # Prefer SHORAD to conserve long-range SAMs if in SHORAD range
            shorad_cand = [c for c in candidates if c[0]["type"] == "SHORAD"]
            if shorad_cand:
                chosen_bat, dist = min(shorad_cand, key=lambda x: x[1])
            else:
                chosen_bat, dist = min(candidates, key=lambda x: x[1])
                
            # Compute interceptor flight time
            interceptor_flight_time = dist / chosen_bat["interceptor_speed"]
            t_impact_interceptor = t_curr + interceptor_flight_time
            
            # Intercept must occur before drone target impact
            if t_impact_interceptor >= t_impact:
                continue
                
            chosen_bat["magazine"] -= 1
            chosen_bat["shots_fired"] += 1
            engaging_battery = chosen_bat["name"]
            engaging_battery_pos = chosen_bat["pos"]
            interception_method = chosen_bat["type"]
            t_fire = t_curr
            t_intercept = t_impact_interceptor
            intercept_pos = traj.evaluate(t_intercept).position
            
            pk = chosen_bat["pk"]
            if np.random.rand() < pk:
                intercepted = True
                chosen_bat["intercepts_confirmed"] += 1
                break
            else:
                continue
                
        if not intercepted:
            target_name = threat["target"]
            target_status[target_name]["destroyed"] = True
            target_status[target_name]["hits"] += 1
            outcome = f"HIT TARGET: {target_name}"
            t_end = t_impact
        else:
            outcome = f"INTERCEPTED BY {engaging_battery} ({interception_method})"
            t_end = t_intercept
            
        duel_results.append({
            "threat_id": threat["id"],
            "wave": threat["wave"],
            "role": threat["role"],
            "color": threat["color"],
            "base": threat["base"],
            "base_pos": threat["base_pos"],
            "route": threat["flank_route"],
            "target": threat["target"],
            "target_pos": threat["target_pos"],
            "rcs": threat["rcs"],
            "t_launch": t_launch,
            "t_fire": t_fire,
            "t_intercept": t_intercept,
            "t_impact": t_impact,
            "t_end": t_end,
            "intercepted": intercepted,
            "assigned_battery": engaging_battery if engaging_battery else "None (Undetected/OutOfRange)",
            "assigned_battery_pos": engaging_battery_pos,
            "intercept_pos": intercept_pos,
            "outcome": outcome,
            "trajectory": threat["trajectory"]
        })
        
    return duel_results, battery_states, target_status

duel_results, battery_states, target_status = simulate_drone_swarm_engagement(
    drone_threats, DEFENDER_BATTERIES, DEFENDING_TARGETS
)
print("Staggered drone swarm simulation completed.")


Staggered drone swarm simulation completed.


In [6]:
# ==============================================================================
# 6. DRONE SWARM ENGAGEMENT TELEMETRY & STATISTICAL REPORT
# ==============================================================================
df_results = pd.DataFrame(duel_results)

df_view = df_results[['threat_id', 'wave', 'role', 'route', 'base', 'assigned_battery', 't_launch', 'intercepted', 'outcome']].copy()
df_view.columns = ['Threat #', 'Wave', 'Role', 'Route', 'Base', 'Engaged Battery', 'Launch (s)', 'Intercepted', 'Tactical Outcome']

print("=" * 105)
print(f"STAGGERED DRONE SWARM ENGAGEMENT REPORT (TOTAL DRONES: {len(df_results)})")
print("=" * 105)
print(df_view.to_string(index=False))

total_intercepted = int(df_results['intercepted'].sum())
total_penetrated = len(df_results) - total_intercepted
pen_rate = (total_penetrated / len(df_results)) * 100.0

print("=" * 105)
print(f"SUMMARY: {total_intercepted} Intercepted ({total_intercepted/len(df_results)*100:.1f}%), {total_penetrated} Penetrated Defense Grid ({pen_rate:.1f}% Penetration Rate)")
print("=" * 105)


STAGGERED DRONE SWARM ENGAGEMENT REPORT (TOTAL DRONES: 30)
 Threat #  Wave                   Role           Route    Base                       Engaged Battery  Launch (s)  Intercepted                                              Tactical Outcome
       10     1      SEAD Radar-Hunter     south_flank Base A2     Battery D1 (Long-Range SAM North)    2.760191         True        INTERCEPTED BY Battery D1 (Long-Range SAM North) (SAM)
        3     1      SEAD Radar-Hunter     south_flank Base A1     Battery D1 (Long-Range SAM North)    4.647073         True        INTERCEPTED BY Battery D1 (Long-Range SAM North) (SAM)
        7     1      SEAD Radar-Hunter     north_flank Base A5     Battery D1 (Long-Range SAM North)    0.303889         True        INTERCEPTED BY Battery D1 (Long-Range SAM North) (SAM)
        6     1      SEAD Radar-Hunter     north_flank Base A3          None (Undetected/OutOfRange)    8.059957        False                   HIT TARGET: Early Warning Radar Array Alpha
 

In [7]:
# ==============================================================================
# 7. BATTERY EXPENDITURE & TARGET DAMAGE AUDIT
# ==============================================================================
df_bats = pd.DataFrame(battery_states)[['name', 'type', 'magazine', 'shots_fired', 'intercepts_confirmed']]
df_bats.columns = ['Battery Name', 'System Class', 'Remaining Magazine', 'Rounds Fired', 'Kills Confirmed']

print("=" * 90)
print("DEFENDER BATTERY MAGAZINE STATUS & SHOT EFFICIENCY")
print("=" * 90)
print(df_bats.to_string(index=False))
print("=" * 90)

destroyed_targets = [k for k, v in target_status.items() if v["destroyed"]]
print(f"\nVital Defending Targets Destroyed by Drone Strikes: {len(destroyed_targets)} / {len(target_status)}")
for dt in destroyed_targets:
    print(f"  - [DESTROYED] {dt} (Direct Hits: {target_status[dt]['hits']}, Strategic Value: {target_status[dt]['value']})")


DEFENDER BATTERY MAGAZINE STATUS & SHOT EFFICIENCY
                           Battery Name System Class  Remaining Magazine  Rounds Fired  Kills Confirmed
      Battery D1 (Long-Range SAM North)          SAM                   0            14               12
       Battery D2 (SHORAD/CIWS Central)       SHORAD                  28             2                2
      Battery D3 (Long-Range SAM South)          SAM                   8             6                6
  Battery D4 (SHORAD North Radar Guard)       SHORAD                  23             1                1
            Battery D5 (SAM Mid-Sector)          SAM                  16             0                0
Battery D6 (SHORAD South Airbase Guard)       SHORAD                  24             0                0

Vital Defending Targets Destroyed by Drone Strikes: 4 / 10
  - [DESTROYED] Early Warning Radar Array Alpha (Direct Hits: 2, Strategic Value: 150.0)
  - [DESTROYED] Forward Radar Picket Charlie (Direct Hits: 4, Strategic 

In [8]:
# ==============================================================================
# 8. PLAYABLE / PAUSABLE 3D STAGGERED DRONE SWARM VISUALIZER
# ==============================================================================

def build_staggered_drone_visualizer(threat_records, battery_list, target_list, n_frames=60, t_total=240.0):
    fig = go.Figure()
    
    # 1. Base Static Elements: Aggressor and Defended Sector Ground Meshes
    base_traces = [
        go.Mesh3d(
            x=[0, 500e3, 500e3, 0], y=[0, 0, THEATER_Y_MAX, THEATER_Y_MAX], z=[0, 0, 0, 0],
            color='rgba(255, 60, 60, 0.22)', name='Aggressor Staging Sector (0-500 km)', hoverinfo='name'
        ),
        go.Mesh3d(
            x=[9500e3, 12000e3, 12000e3, 9500e3], y=[0, 0, THEATER_Y_MAX, THEATER_Y_MAX], z=[0, 0, 0, 0],
            color='rgba(0, 180, 255, 0.20)', name='Defended Vital Sector (>= 10,000 km)', hoverinfo='name'
        ),
        # Aggressor Launch Bases
        go.Scatter3d(
            x=[b['pos'][0] for b in AGGRESSOR_LAUNCH_BASES],
            y=[b['pos'][1] for b in AGGRESSOR_LAUNCH_BASES],
            z=[b['pos'][2] for b in AGGRESSOR_LAUNCH_BASES],
            mode='markers+text',
            marker=dict(size=7, color='#ff4444', symbol='circle'),
            text=[b['id'] for b in AGGRESSOR_LAUNCH_BASES],
            textposition='top center',
            name='Aggressor Drone Bases'
        )
    ]
    
    # Defender Batteries
    for b in battery_list:
        base_traces.append(go.Scatter3d(
            x=[b['pos'][0]], y=[b['pos'][1]], z=[b['pos'][2]],
            mode='markers+text',
            marker=dict(size=8, color='#00d0ff' if b['type'] == 'SAM' else '#00ffaa', symbol='diamond'),
            text=[b['name'].split()[1]],
            textposition='bottom center',
            name=b['name'],
            hovertext=f"{b['name']} ({b['type']})",
            showlegend=False
        ))
        
    # Vital Targets
    for t in target_list:
        is_dest = target_status[t['name']]['destroyed']
        base_traces.append(go.Scatter3d(
            x=[t['pos'][0]], y=[t['pos'][1]], z=[t['pos'][2]],
            mode='markers+text',
            marker=dict(size=7, color='#ff2244' if is_dest else '#00ff66', symbol='square'),
            text=[t['name'].split()[0]],
            textposition='top center',
            name=t['name'],
            hovertext=f"{t['name']} ({'DESTROYED' if is_dest else 'INTACT'})",
            showlegend=False
        ))
        
    # Static faint guide paths showing complete spline curves
    for d in threat_records:
        traj = d['trajectory']
        t_sample = np.linspace(d['t_launch'], d['t_impact'], 40)
        pts = np.array([traj.evaluate(t).position for t in t_sample])
        base_traces.append(go.Scatter3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            mode='lines',
            line=dict(color=d['color'], width=1.5, dash='dot'),
            opacity=0.20,
            showlegend=False,
            hoverinfo='none'
        ))
        
    # Initial Frame 0 Active Traces per Drone Threat
    for d in threat_records:
        bp = d['base_pos']
        col = d['color']
        base_traces.append(go.Scatter3d(x=[bp[0]], y=[bp[1]], z=[bp[2]], mode='lines', line=dict(color=col, width=3), showlegend=False))
        base_traces.append(go.Scatter3d(x=[bp[0]], y=[bp[1]], z=[bp[2]], mode='markers', marker=dict(size=5, color=col, symbol='circle'), showlegend=False))
        bat_pos = d['assigned_battery_pos'] if d['assigned_battery_pos'] is not None else bp
        base_traces.append(go.Scatter3d(x=[bat_pos[0]], y=[bat_pos[1]], z=[bat_pos[2]], mode='lines', line=dict(color='#00ffff', width=2), showlegend=False))
        base_traces.append(go.Scatter3d(x=[bat_pos[0]], y=[bat_pos[1]], z=[bat_pos[2]], mode='markers', marker=dict(size=4, color='#00ffff', symbol='diamond'), showlegend=False))
        base_traces.append(go.Scatter3d(x=[bp[0]], y=[bp[1]], z=[bp[2]], mode='markers', marker=dict(size=1, color='rgba(0,0,0,0)'), showlegend=False))
        
    for tr in base_traces:
        fig.add_trace(tr)
        
    n_static = 3 + len(battery_list) + len(target_list) + len(threat_records)
    
    # 2. Build Staggered Animation Frames
    time_frames = np.linspace(0.0, t_total, n_frames)
    frames = []
    
    for k, t_curr in enumerate(time_frames):
        frame_data = []
        for d in threat_records:
            traj = d['trajectory']
            t_l = d['t_launch']
            t_end = d['t_end']
            col = d['color']
            
            # --- Drone Motion ---
            if t_curr < t_l:
                bp = d['base_pos']
                dx, dy, dz = [bp[0]], [bp[1]], [bp[2]]
                hx, hy, hz = [bp[0]], [bp[1]], [bp[2]]
                head_size = 0.1
            elif t_curr <= t_end:
                sub_t = np.linspace(t_l, t_curr, max(2, int(20 * (t_curr - t_l)/(t_end - t_l + 1e-3))))
                pts = np.array([traj.evaluate(t).position for t in sub_t])
                dx, dy, dz = pts[:, 0].tolist(), pts[:, 1].tolist(), pts[:, 2].tolist()
                hx, hy, hz = [dx[-1]], [dy[-1]], [dz[-1]]
                head_size = 6
            else:
                sub_t = np.linspace(t_l, t_end, 20)
                pts = np.array([traj.evaluate(t).position for t in sub_t])
                dx, dy, dz = pts[:, 0].tolist(), pts[:, 1].tolist(), pts[:, 2].tolist()
                hx, hy, hz = [dx[-1]], [dy[-1]], [dz[-1]]
                head_size = 0.1
                
            frame_data.append(go.Scatter3d(x=dx, y=dy, z=dz, mode='lines', line=dict(color=col, width=3 if t_curr <= t_end else 1)))
            frame_data.append(go.Scatter3d(x=hx, y=hy, z=hz, mode='markers', marker=dict(size=head_size, color=col, symbol='circle')))
            
            # --- Interceptor Motion ---
            if d['t_fire'] is not None and d['assigned_battery_pos'] is not None and t_curr >= d['t_fire']:
                bat_pos = d['assigned_battery_pos']
                int_pos = d['intercept_pos'] if d['intercept_pos'] is not None else d['target_pos']
                t_f = d['t_fire']
                t_int = d['t_intercept'] if d['t_intercept'] is not None else t_end
                
                frac = min(1.0, max(0.0, (t_curr - t_f) / (t_int - t_f + 1e-3)))
                curr_int_pos = bat_pos + frac * (int_pos - bat_pos)
                
                ix = [bat_pos[0], curr_int_pos[0]]
                iy = [bat_pos[1], curr_int_pos[1]]
                iz = [bat_pos[2], curr_int_pos[2]]
                ih_size = 5 if frac < 1.0 else 0.1
            else:
                bp = d['base_pos']
                ix, iy, iz = [bp[0]], [bp[1]], [bp[2]]
                ih_size = 0.1
                
            frame_data.append(go.Scatter3d(x=ix, y=iy, z=iz, mode='lines', line=dict(color='#00ffff', width=2)))
            frame_data.append(go.Scatter3d(x=[ix[-1]], y=[iy[-1]], z=[iz[-1]], mode='markers', marker=dict(size=ih_size, color='#00ffff', symbol='diamond')))
            
            # --- Burst / Detonation Event ---
            if t_curr >= t_end:
                if d['intercepted'] and d['intercept_pos'] is not None:
                    bx, by, bz = [d['intercept_pos'][0]], [d['intercept_pos'][1]], [d['intercept_pos'][2]]
                    b_color, b_sym, b_sz = '#00ffff', 'cross', 8
                else:
                    bx, by, bz = [d['target_pos'][0]], [d['target_pos'][1]], [d['target_pos'][2]]
                    b_color, b_sym, b_sz = '#ff0033', 'x', 10
            else:
                bx, by, bz = [d['base_pos'][0]], [d['base_pos'][1]], [d['base_pos'][2]]
                b_color, b_sym, b_sz = 'rgba(0,0,0,0)', 'circle', 0.1
                
            frame_data.append(go.Scatter3d(x=bx, y=by, z=bz, mode='markers', marker=dict(size=b_sz, color=b_color, symbol=b_sym)))
            
        frames.append(go.Frame(data=frame_data, traces=list(range(n_static, n_static + len(frame_data))), name=f"f_{k}"))
        
    fig.frames = frames
    
    # 3. Interactive Play / Pause & Time Slider Controls
    slider_steps = []
    for k, t_curr in enumerate(time_frames):
        slider_steps.append({
            "args": [[f"f_{k}"], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
            "label": f"{t_curr:.0f}s",
            "method": "animate"
        })
        
    fig.update_layout(
        title="<b>Staggered Drone Swarm Theater Duel</b> (C2 Waypoint Splines & Layered Air Defense)",
        title_font=dict(size=16, color='#00d0ff'),
        paper_bgcolor="#0e1117",
        plot_bgcolor="#0e1117",
        scene=dict(
            xaxis=dict(title="Downrange X (km)", range=[0, THEATER_X_MAX], backgroundcolor="#080b10", gridcolor="#1a2233"),
            yaxis=dict(title="Crossrange Y (km)", range=[0, THEATER_Y_MAX], backgroundcolor="#080b10", gridcolor="#1a2233"),
            zaxis=dict(title="Altitude Z (m)", range=[0, 3000], backgroundcolor="#080b10", gridcolor="#1a2233"),
            aspectmode="manual",
            aspectratio=dict(x=3.0, y=2.5, z=0.5),
            camera=dict(eye=dict(x=-1.6, y=-1.8, z=0.9))
        ),
        updatemenus=[{
            "type": "buttons",
            "showactive": False,
            "x": 0.05, "y": 0.0,
            "xanchor": "left", "yanchor": "top",
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [None, {"frame": {"duration": 40, "redraw": True}, "fromcurrent": True, "transition": {"duration": 0}}]
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "transition": {"duration": 0}}]
                }
            ]
        }],
        sliders=[{
            "steps": slider_steps,
            "x": 0.22, "y": 0.0,
            "len": 0.75,
            "xanchor": "left", "yanchor": "top",
            "currentvalue": {"font": {"size": 13, "color": "#00d0ff"}, "prefix": "Simulation Time: ", "visible": True, "xanchor": "right"},
            "pad": {"b": 10, "t": 10}
        }],
        margin=dict(l=0, r=0, b=0, t=40),
        height=800
    )
    
    return fig

fig_drone_duel = build_staggered_drone_visualizer(duel_results, DEFENDER_BATTERIES, DEFENDING_TARGETS, n_frames=60, t_total=240.0)
fig_drone_duel.show()
